In [1]:
import pandas as pd

df = pd.read_csv('C:/OPS/hybrid_filtering/0507_userindex.csv')
mart = pd.read_csv('C:/OPS/hybrid_filtering/vod_mart_processed.csv')

C:\Users\user\AppData\Local\Temp\ipykernel_19036\3971008490.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  mart = pd.read_csv('C:/OPS/hybrid_filtering/vod_mart_processed.csv')


In [2]:
df.drop(columns = {'sha2_hash', 'user_label', 'asset_nm', 'disp_rtm'}, inplace = True)

In [3]:
# mart에서 필요한 컬럼만 선택
mart_subset = mart[['asset_id', 'super_asset_nm', 'category_l2', 'genre', 'disp_rtm']]

# df와 mart_subset을 asset_id를 기준으로 병합
df = pd.merge(df, mart_subset, on='asset_id', how='left')

In [4]:
mart_subset = mart[['asset_id', 'smry']]

# df와 mart_subset을 asset_id를 기준으로 병합
df = pd.merge(df, mart_subset, on='asset_id', how='left')

In [5]:
mart_subset = mart[['asset_id', 'ct_cl']]

# df와 mart_subset을 asset_id를 기준으로 병합
df = pd.merge(df, mart_subset, on='asset_id', how='left')

In [6]:
import re

# 괄호와 특수문자를 제거하는 함수 정의
def clean_text(text):
    if pd.isna(text):  # NaN 값 처리
        return text
    # 괄호와 괄호 안의 내용 제거
    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'\[[^\]]*\]', ' ', text)
    # 특수문자 제거 (한글, 영문, 숫자만 남김)
    text = re.sub(r'[^\w\s가-힣]', ' ', text)
    # 공백 제거
    text = text.strip()
    return text

# 각 컬럼에 대해 정제 적용
df['super_asset_nm'] = df['super_asset_nm'].apply(clean_text)
df['genre'] = df['genre'].apply(clean_text)
df['smry'] = df['smry'].apply(clean_text)
df['ct_cl'] = df['ct_cl'].apply(clean_text)

## 파생변수 view_ratio 생성

In [7]:
# use_tms를 disp_rtm으로 나누어 비율 계산
df['disp_rtm'].astype(int)
df['view_ratio'] = (df['use_tms'] / df['disp_rtm']).clip(upper = 1)

In [8]:
df['view_ratio'].isna().sum()

7578

In [9]:
df.dropna(subset = 'view_ratio', inplace = True)

### 주기성 반영 파생변수 생성

In [10]:
import numpy as np
import pandas as pd

# 날짜 컬럼을 datetime으로 변환
df['strt_dt_dt'] = pd.to_datetime(df['strt_dt_dt'])

# 연도, 월, 일, 주차, 시간 추출
df['weekday'] = df['strt_dt_dt'].dt.weekday  # 0~6
df['hour'] = df['strt_dt_dt'].dt.hour

# sin-cos 주기적 변환 함수
def encode_cyclic(df, col, max_val):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

# 월 (1~12), 일 (1~31), 요일 (0~6), 시간 (0~23)
df = encode_cyclic(df, 'weekday', 7)
df = encode_cyclic(df, 'hour', 24)

df

,use_tms,asset_id,strt_dt_dt,user_index,super_asset_nm,category_l2,genre,disp_rtm,smry,ct_cl,view_ratio,weekday,hour,weekday_sin,weekday_cos,hour_sin,hour_cos
0,2880,M5047991LFOJ44245901,2023-05-03 22:38:06,0,전국민민원해결프로젝트 일꾼의탄생,(HD)KBS 시사교양,시사 교양,2880.0,단결 특전사 707부대 출신 최영재 오늘 신입 일꾼으로 민원 해결을 명 받았습니...,TV 시사 교양,1.000000,2,22,0.974928,-0.222521,-5.000000e-01,8.660254e-01
1,926,M5164421LFOI39723501,2023-05-03 17:24:13,1,나 혼자산다,(HD)MBC 연예오락,연예 오락,5580.0,나 혼자 산다 육체미 소동 편 피지컬 폭발 육체미 넘치는 무지개 회원들의 하루 ...,TV 연예 오락,0.165950,2,17,0.974928,-0.222521,-9.659258e-01,-2.588190e-01
2,500,M0191290LSGK72289001,2023-05-03 16:09:53,2,명성황후,KBS구작,미니시리즈,3660.0,명성황후를 시해한 미우라는 이번 일이 훈련대와 대원군에 의해서 자행된 것으로 조작하...,TV드라마,0.136612,2,16,0.974928,-0.222521,-8.660254e-01,-5.000000e-01
3,0,M5092600LFOK59331601,2023-05-03 23:39:48,3,꼬리에꼬리를무는그날이야기,(HD)SBS 시사교양,시사 교양,4320.0,필사의 도주 벼랑 끝에 선 사람들 1972년 8월 19일 충북 단양의 남한강 유...,TV 시사 교양,0.000000,2,23,0.974928,-0.222521,-2.588190e-01,9.659258e-01
4,5580,M5066112LFOJ84983101,2023-05-03 18:48:46,4,골 때리는 그녀들,(HD)SBS 연예오락,연예 오락,5580.0,차원이 다른 경기가 시작된다 필드를 아우르는 초특급 에이스 군단 총출동 22명의...,TV 연예 오락,1.000000,2,18,0.974928,-0.222521,-1.000000e+00,-1.836970e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17175098,12,M4789223LSGJ69166601,2023-05-14 20:40:29,8419,헤이지니 럭키강이 시즌14 스페셜,만화동산,기타,1140.0,1 레고 음식 vs 실제 음식 레고 음식 vs 실제 음식 진짜보다 더 진짜 같...,키즈,0.010526,6,20,-0.781831,0.623490,-8.660254e-01,5.000000e-01
17175099,4920,M5143337LFOL00172201,2023-05-14 12:17:02,8416,한문철의 블랙박스 리뷰,JTBC시사교양,교양다큐,4920.0,한블리에도 찾아온 크리스마스 연말연시만 되면 폭증하는 도로 위 묻지 마 살인 음...,TV 시사 교양,1.000000,6,12,-0.781831,0.623490,1.224647e-16,-1.000000e+00
17175100,2769,M4923883LSGJ81263001,2023-05-14 18:12:17,37101,심야괴담회,MBC구작,연예 오락,4380.0,외로운 당신에게 어둑시니의 초대장이 도착했습니다 이곳에서 당신의 무서운 이야기를 ...,TV 연예 오락,0.632192,6,18,-0.781831,0.623490,-1.000000e+00,-1.836970e-16
17175101,694,M5042751LFOJ31748801,2023-05-14 13:29:21,67596,런닝맨,(HD)SBS 연예오락,연예 오락,5220.0,결정적 한 빵 근본 있는 알싸한 주먹에 도전장 내민 파이터 주우재 변우석 박경...,TV 연예 오락,0.132950,6,13,-0.781831,0.623490,-2.588190e-01,-9.659258e-01


In [11]:
df.drop(columns = {'use_tms', 'strt_dt_dt', 'disp_rtm', 'category_l2', 'weekday', 'hour'}, inplace = True)

In [12]:
df.isna().sum()

asset_id          0
user_index        0
super_asset_nm    0
genre             0
smry              0
ct_cl             0
view_ratio        0
weekday_sin       0
weekday_cos       0
hour_sin          0
hour_cos          0
dtype: int64

In [13]:
df.to_csv('C:/OPS/hybrid_filtering/processed_data.csv', index = False)

# ALS

In [ ]:
from scipy import sparse

USER_COL  = "user_index"   # 필요 시 수정
ITEM_COL  = "asset_id"
RATING_COL = "view_ratio"   # 0~5 범위라 가정

# 범주형 → 순차 인덱스
user_codes, user_uniques = pd.factorize(df[USER_COL])
item_codes, item_uniques = pd.factorize(df[ITEM_COL])

alpha = 40.0                      # 명시적 평점을 implicit confidence 로 키우는 계수
data  = (df[RATING_COL] * alpha).astype(np.float32)

interaction_coo = sparse.coo_matrix(
    (data, (user_codes, item_codes)),
    shape=(len(user_uniques), len(item_uniques)),
    dtype=np.float32
)

print(f"희소 행렬 크기: {interaction_coo.shape}")
print(f"데이터 밀도: {interaction_coo.nnz / (interaction_coo.shape[0]*interaction_coo.shape[1]):.6f}")

In [ ]:
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

# implicit 패키지는 (item × user) CSR 형태를 기대
item_user_csr = interaction_coo.T.tocsr()

als = AlternatingLeastSquares(
    factors=64,
    regularization=0.015,
    iterations=20,
    calculate_training_loss=True,
    use_gpu=False          # CUDA 가능 환경이면 True
)

# 진행률 표시용 함수 덮어쓰기 (선택)
als.progress = lambda x, **k: tqdm(total=x)  # tqdm 적용

als.fit(item_user_csr)
print("✅ 학습 완료")

In [ ]:
# --- 학습 완료 후 ---
if hasattr(als, "progress"):
    del als.progress        # picklable 하지 않은 속성 삭제

In [ ]:
import numpy as np
import pickle, pathlib

SAVE_DIR = pathlib.Path("artifacts")
SAVE_DIR.mkdir(exist_ok=True)

# 1) 잠재 요인 행렬만 .npy 로 저장 ────────────────────────────
np.save(SAVE_DIR / "C:/hybrid_filtering/user_factors.npy", als.user_factors)
np.save(SAVE_DIR / "C:/hybrid_filtering/item_factors.npy", als.item_factors)

# 2) 매핑 정보까지 묶어 pkl 로 직렬화(선택) ────────────────
payload = {
    "model": als,             # 전체 모델 객체
    "user_map": user_uniques, # factorize 때 나온 시리즈
    "item_map": item_uniques,
}
with open(SAVE_DIR / "als_full.pkl", "wb") as f:
    pickle.dump(payload, f)

print("✅ user_factors / item_factors .npy 저장 완료")
print("✅ 모델·매핑 포함 pkl 저장 완료")

# 컨텐츠 기반 필터링 아이템 프로필(Okt + TF-IDF)

In [ ]:
df.columns

In [ ]:
TEXT_COLS = {
    "super_asset_nm": "[TITLE]",   # 제목
    "genre":          "[GENRE]",   # 장르(복수 가능)
    "category_l2":    "[CAT]",     # 카테고리
    "smry":           "[PLOT]",    # 줄거리/요약
}

NUM_COLS   = [                                # 수치형
    "month_sin", "month_cos",
    "day_sin", "day_cos",
    "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    'rating_qt'
]

df[list(TEXT_COLS)].head()

In [ ]:
from mecab import MeCab
mecab = MeCab()

TAG_KEEP_MECAB = {'NNG', 'NNP', 'VV', 'VA', 'XR'}  # 일반명사, 고유명사, 동사, 형용사, 어근

def mecab_tokenizer(text: str):
    return [
        morph for morph, pos in mecab.pos(text)
        if pos in TAG_KEEP_MECAB
    ]

In [ ]:
# 2. 텍스트 컬럼 합치기
# ------------------------------------------------------------
def concat_text(row):
    parts = []
    for col, prefix in TEXT_COLS.items():
        txt = str(row[col]) if pd.notna(row[col]) else ""
        if txt:
            parts.append(f"{prefix} {txt}")
    return " ".join(parts)

corpus = df.apply(concat_text, axis=1)

In [ ]:
# 3. TF-IDF (Mecab 기반)
# ------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    tokenizer=mecab_tokenizer,
    ngram_range=(1, 2),
    max_features=10_000,
    min_df=3,
)
X_text = tfidf.fit_transform(corpus)
print("TF-IDF :", X_text.shape)

# (선택) 필드 가중치
weights = { "[TITLE]": 2.0, "[GENRE]": 3.0, "[CAT]": 2.5, "[PLOT]": 1.0 }
vocab, scale = tfidf.vocabulary_, np.ones(X_text.shape[1])
for tok, w in weights.items():
    key = tok.lower()
    if key in vocab:
        scale[vocab[key]] = w
X_text = X_text.multiply(scale)